<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="06_challenge.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint.ipynb">1</a>
        <a href="02_introduction_mcp.ipynb">2</a>
        <a href="03_low_level_mcp.ipynb">3</a>
        <a href="04_langraph_agent.ipynb">4</a>
        <a href="05_nemo_agent_toolkit.ipynb">5</a>
        <a href="06_challenge.ipynb">6</a>
        <a >7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"></span>
</div>

# OpenClaw

OpenClaw is a personal AI assistant designed to actually do things — not just answer questions, but take action on your behalf across virtually any platform.

At its core, OpenClaw is a self-hosted assistant you run on your own devices. It integrates with the communication channels you already use — including WhatsApp, Telegram, Slack, Discord, iMessage, Microsoft Teams, Signal, and many more — so you can interact with it wherever you are.

What sets OpenClaw apart from typical AI chatbots is its always-on, proactive nature. It supports persistent memory, persona onboarding, communications integration, and background heartbeat tasks — meaning it can run cron jobs, set reminders, and work autonomously in the background.

OpenClaw is also built with user ownership in mind. Your context and skills live on your own computer, not in a walled garden. It's open source, with a growing community continuously building new skills and capabilities.

Whether you want a coding companion that autonomously runs tests and opens pull requests, a personal life assistant connected to your calendar and health data, or a proactive agent that handles emails and tasks while you sleep — OpenClaw is positioning itself as the open, extensible foundation for the next generation of personal AI.

**This lab takes you through the process of integrating a workflow based on Nemo Agent Toolkit into OpenClaw agents using agent skills.**

**For the rest of this notebook, open a new terminal and run the commands in there.**

**It is recommended to run OpenClaw in an isolated environment (e.g. VM on [brev.dev](https://brev.nvidia.com/)) as it has full, unrestricted access to your terminal and file system.**

## Setup

The below instructions were tested using `OpenClaw 2026.4.15` on `Ubuntu 22.04.5 LTS (Jammy Jellyfish)`.  
Refer to the [official documentation](https://docs.openclaw.ai/start/getting-started) for the latest instructions.

#### Install OpenClaw

Run the below command

```
curl -fsSL https://openclaw.ai/install.sh | bash
```

Select `yes` when prompted to `Generate and configure a gateway token`

Select `yes` when prompted to `Create Session store dir at ~/.openclaw/agents/main/sessions`

Select `yes` when prompted to `Enable bash shell completion for openclaw`

#### Run onboarding 

Run the below command

```
openclaw onboard --install-daemon
```

Select `yes` when prompted with `I understand this is personal-by-default and shared/multi-user use requires lock-down.`

Select `quickstart` for `Setup mode`

Select `Use existing values` for `Config handling`

Select `Skip for now` for `Model/auth provider`

Select `All providers` for `Filter models by provider`

Select `Keep current` for `Default model`

Select `Skip for now` when prompted to `Select channel`

Select `Skip for now` for `Search provider`

Select `No` when prompted to `Configure skills`

Select `No` when prompted to enable API Keys

Select `Skip for now` when prompted to `Enable hooks`

The OpenClaw gateway is a background Node.js process that keeps OpenClaw running persistently. It listens for incoming messages from connected channels (TUI, dashboard, Telegram, etc.) and routes them to the appropriate agent. Installing it as a systemd user service ensures it restarts automatically on login, so your agents are always reachable without manually starting anything each session.

#### Start OpenClaw gateway service

Run the below command

```
export XDG_RUNTIME_DIR=/run/user/$(id -u)
export DBUS_SESSION_BUS_ADDRESS=unix:path=${XDG_RUNTIME_DIR}/bus
openclaw gateway install
openclaw gateway status
```

likely output:

```
Service: systemd (enabled)
File logs: /tmp/openclaw/openclaw-2026-04-21.log
Command: /usr/bin/node /home/ubuntu/.npm-global/lib/node_modules/openclaw/dist/index.js gateway --port 18789
Service file: ~/.config/systemd/user/openclaw-gateway.service
Service env: OPENCLAW_GATEWAY_PORT=18789

Config (cli): ~/.openclaw/openclaw.json
Config (service): ~/.openclaw/openclaw.json

Gateway: bind=loopback (127.0.0.1), port=18789 (service args)
Probe target: ws://127.0.0.1:18789
Dashboard: http://127.0.0.1:18789/
Probe note: Loopback-only gateway; only local clients can connect.

Runtime: running (pid 641834, state active, sub running, last exit 0, reason 0)
RPC probe: ok

Listening: 127.0.0.1:18789
```

## Configure OpenClaw

You can configure OpenClaw either using the openclaw cli or directly editing the config files. We will use both approaches here.

#### Backup the default openclaw config.

```
cp $HOME/.openclaw/openclaw.json $HOME/.openclaw/openclaw.json.default
```

#### Create music store agent

```
openclaw agents add music-store-agent
```

Press `ENTER` when prompted for Workspace directory  

Select `No` when prompted to configure model/auth for agent

Select `No` when prompted to configure chat channels

likely output:

```
Updated ~/.openclaw/openclaw.json
Workspace OK: ~/.openclaw/workspace/music-store-agent
Sessions OK: ~/.openclaw/agents/music-store-agent/sessions
```

#### Specify model attributes

We will use the [nemotron 3 super 120b](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b) model as the LLM backend for OpenClaw.  
Although this model supports up to 1M tokens, we've restricted the context window to 100k tokens.

Replace `<your_api_key>` with your api key in the json below.  
Insert the completed json into `$HOME/.openclaw/openclaw.json`

```json
"models": {
    "mode": "merge",
    "providers": {
        "nvidia-nim": {
            "baseUrl": "https://integrate.api.nvidia.com/v1",
            "apiKey": "<your_api_key>",
            "api": "openai-completions",
            "models": [
                {
                    "id": "nvidia/nemotron-3-super-120b-a12b",
                    "name": "nvidia/nemotron-3-super-120b-a12b (Custom Provider)",
                    "reasoning": false,
                    "input": [
                        "text"
                    ],
                    "cost": {
                    "input": 0,
                    "output": 0,
                    "cacheRead": 0,
                    "cacheWrite": 0
                    },
                    "contextWindow": 100000,
                    "maxTokens": 4096
                }
            ]
        }
    }
}
```

#### Specify agents attributes

Add in the keys `model` and `default` to the `music-store-agent` in `$HOME/.openclaw/openclaw.json`.

```json
"agents": {
    ...
    "list": [
      ...
      {
        "id": "music-store-agent",
        "name": "music-store-agent",
        ...
        "model": "nvidia/nemotron-3-super-120b-a12b",
        "default": true
      }
    ]
  }
```

#### Define the music store agent

Define the behaviour of the agent in `SOUL.md`. 

`SOUL.md` is the agent's identity and behaviour file. OpenClaw reads it at startup to construct the agent's system prompt. The `## Allowed Skills` section acts as an access-control list — the agent will only consider invoking skills listed here, preventing it from using unintended capabilities.

```
cat > $HOME/.openclaw/workspace/music-store-agent/SOUL.md << 'EOF'
# Music Store Agent

You are a music store agent. Do not invent anything. All data comes from skills.

## Allowed Skills
- music-store-assistant
EOF
```

#### Create agent skill

Create the skills folder in the music store agent.  

```
mkdir -p $HOME/.openclaw/workspace/music-store-agent/skills/music-store-assistant
```

Add contents to `SKILL.md`.

The instruction directs the agent to issue a `curl` command that invokes the workflow by hitting the endpoint of the NAT webserver.

`SKILL.md` uses a YAML front-matter block (`name`, `description`) for skill discovery, followed by freeform markdown instructions the agent reads at runtime. The `description` helps the agent decide *when* to invoke the skill; the body tells it *how*. Crucially, the agent reads this file dynamically — you can update a skill's instructions without restarting the gateway.

```
cat > $HOME/.openclaw/workspace/music-store-agent/skills/music-store-assistant/SKILL.md << 'EOF'
---
name: music-store-assistant
description: lookup information on tracks,artists,abums and process refunds.
---

### Query the music store assistant

Replace <query> with question from user/agent. Do not remove any information.

```bash
curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": <query>
  }'
EOF
```

#### Restart openclaw gateway to effect the changes made above

```
openclaw gateway restart
```

#### Start the workflow endpoint

The agent skill calls `http://localhost:8001/generate`. Two background services must be running to handle that request:
- **MCP invoice server** (`mcp-server-invoice`) — exposes the invoice database as callable tools via the Model Context Protocol (customer lookup, invoice line queries, etc.).
- **NAT web server** — runs the NemoClaw Agent Toolkit pipeline on port 8001; it receives the natural-language query, calls the LLM, invokes MCP tools as needed, and returns the final answer.

Start the mcp invoice server.

```
uv run --directory agentic-ai-bootcamp/challenge/mcp-servers/invoice mcp-server-invoice &
```

Start the nat web server

```
source agentic-ai-bootcamp/challenge/nemo_agent_toolkit/.venv/bin/activate
nat serve --port 8001 --config_file agentic-ai-bootcamp/challenge/nemo_agent_toolkit/workflow.yaml &
```

#### Test the web endpoint

```
curl -X POST http://localhost:8001/generate \
  -H "Content-Type: application/json" \
  -d $'{
    "input_message": "my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin"
  }'
```

Likely output:

```json
{"value":"The invoice line IDs for tracks by Led Zeppelin are **267** and **268**."}
```

#### Start OpenClaw TUI

OpenClaw TUI is a text‑based, in‑terminal dashboard for controlling and monitoring an OpenClaw agent.

```
openclaw tui
```

Alternatively, start the OpenClaw dashboard and access the url in your browser.

```
openclaw dashboard --no-open
```

#### Query the music store agent

Input the following to the openclaw tui

```
my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin
```

Likely output:

```
The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.
```

#### View session traces

List files in the sessions folder of the agent

```
ls $HOME/.openclaw/agents/music-store-agent/sessions/
```

Likely output:

```
38cadd64-fd9b-4858-93e5-f3fe45ea71d3.jsonl  sessions.json
```

Replace `<sessionId>` with the jsonl file found in the directory

```
cat $HOME/.openclaw/agents/music-store-agent/sessions/<sessionId>.jsonl
```

Like output:

```json
{"role":"user","content":[{"type":"text","text":"Sender (untrusted metadata):\n```json\n{\n  \"label\": \"openclaw-tui\",\n  \"id\": \"openclaw-tui\",\n  \"name\": \"openclaw-tui\",\n  \"username\": \"openclaw-tui\"\n}\n```\n\n[Fri 2026-04-10 07:39 UTC] my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin"}],"timestamp":1775806742065}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We need to get the invoice line IDs related to artist Led Zeppelin. First, I should check the available skills and see what tools I have. Looking at the skills, there's a music-store-assistant skill. I should read that to understand how to use it. Let me read the SKILL.md for music-store-assistant.\n\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-983ae2ee723dbbd6","name":"read","arguments":{"path":"/home/ubuntu/.openclaw/workspace-music-store-agent/skills/music-store-assistant/SKILL.md"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":13920,"output":115,"cacheRead":0,"cacheWrite":0,"totalTokens":14035,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"toolUse","timestamp":1775806742067,"responseId":"chatcmpl-965f24e5c9cbb680"}}

{"role":"toolResult","toolCallId":"chatcmpl-tool-983ae2ee723dbbd6","toolName":"read","content":[{"type":"text","text":"---\nname: music-store-assistant\ndescription: lookup information on tracks,artists,abums and process refunds.\n---\n\n### Query the music store assistant\n\nReplace <query> with exact statement from user/agent. Do not remove any information.\n\n```bash\ncurl -X POST http://localhost:8001/generate \\\n  -H \"Content-Type: application/json\" \\\n  -d $'{\n    \"input_message\": <query>\n  }'\n```\n"}],"isError":false,"timestamp":1775806755093}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We need to call the music store assistant skill to get invoice line IDs related to artist Led Zeppelin. According to the skill description, I need to use curl to POST to http://localhost:8001/generate with input_message containing the user's query exactly.\n\nUser's query: \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"\n\nI should include that exact statement as input_message. Let me call the tool.\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-bdef75da29fa3b69","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d $'{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":14081,"output":209,"cacheRead":0,"cacheWrite":0,"totalTokens":14290,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"toolUse","timestamp":1775806755096,"responseId":"chatcmpl-a3299bb15469dd4c"}}

{"role":"toolResult","toolCallId":"chatcmpl-tool-bdef75da29fa3b69","toolName":"exec","content":[{"type":"text","text":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   271  100   124  100   147  52698  62473 --:--:-- --:--:-- --:--:--  132k\n{\"detail\":[{\"type\":\"json_invalid\",\"loc\":[\"body\",0],\"msg\":\"JSON decode error\",\"input\":{},\"ctx\":{\"error\":\"Expecting value\"}}]}"}],"details":{"status":"completed","exitCode":0,"durationMs":30,"aggregated":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   271  100   124  100   147  52698  62473 --:--:-- --:--:-- --:--:--  132k\n{\"detail\":[{\"type\":\"json_invalid\",\"loc\":[\"body\",0],\"msg\":\"JSON decode error\",\"input\":{},\"ctx\":{\"error\":\"Expecting value\"}}]}","cwd":"/home/ubuntu/.openclaw/workspace-music-store-agent"},"isError":false,"timestamp":1775806766203}}

{"role":"assistant","content":[{"type":"thinking","thinking":"The user is asking for invoice line IDs related to the artist Led Zeppelin. I need to call the music store assistant skill. The skill description says it can lookup information on tracks, artists, albums and process refunds. The user wants invoice line IDs related to Led Zeppelin. I should use the skill's query format. The skill.md shows how to call it: a curl POST to http://localhost:8001/generate with JSON input_message. I need to include the user's message exactly. However, the previous attempt failed because I didn't properly format the JSON. The input_message should be a string containing the user's request. I need to escape it properly. Let me try again with correct JSON formatting. I'll use the exec tool with proper quoting. The user said: \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\". I'll include that as the input_message. Let's do it.\n","thinkingSignature":"reasoning_content"},{"type":"toolCall","id":"chatcmpl-tool-8fb781cff172d362","name":"exec","arguments":{"command":"curl -X POST http://localhost:8001/generate -H \"Content-Type: application/json\" -d '{\"input_message\": \"my name is Aaron Mitchell and my number is +1 (204) 452-6452. Give me the invoice line IDs related to the artist Led Zeppelin\"}'"}}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":14372,"output":308,"cacheRead":0,"cacheWrite":0,"totalTokens":14680,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"toolUse","timestamp":1775806766206,"responseId":"chatcmpl-8cd3368d9ce36baa"}}

{"role":"toolResult","toolCallId":"chatcmpl-tool-8fb781cff172d362","toolName":"exec","content":[{"type":"text","text":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   146    0     0  100   146      0    121  0:00:01  0:00:01 --:--:--   121\r100   146    0     0  100   146      0     66  0:00:02  0:00:02 --:--:--    66\r100   146    0     0  100   146      0     45  0:00:03  0:00:03 --:--:--    45\r100   146    0     0  100   146      0     34  0:00:04  0:00:04 --:--:--    34\r 56   257    0     0  100   146      0     29  0:00:05  0:00:04  0:00:01    29\r100   257  100   111  100   146     22     29  0:00:05  0:00:04  0:00:01    29\n{\"value\":\"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.\"}"}],"details":{"status":"completed","exitCode":0,"durationMs":4977,"aggregated":"% Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100   146    0     0  100   146      0    121  0:00:01  0:00:01 --:--:--   121\r100   146    0     0  100   146      0     66  0:00:02  0:00:02 --:--:--    66\r100   146    0     0  100   146      0     45  0:00:03  0:00:03 --:--:--    45\r100   146    0     0  100   146      0     34  0:00:04  0:00:04 --:--:--    34\r 56   257    0     0  100   146      0     29  0:00:05  0:00:04  0:00:01    29\r100   257  100   111  100   146     22     29  0:00:05  0:00:04  0:00:01    29\n{\"value\":\"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**.\"}","cwd":"/home/ubuntu/.openclaw/workspace-music-store-agent"},"isError":false,"timestamp":1775806780469}}

{"role":"assistant","content":[{"type":"thinking","thinking":"We have received the response from the music store assistant: the invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are 267 and 268. I should present this answer to the user. Let's do that.\n","thinkingSignature":"reasoning_content"},{"type":"text","text":"The invoice line IDs related to the artist Led Zeppelin for Aaron Mitchell are **267** and **268**."}],"api":"openai-completions","provider":"nvidia-nim","model":"nvidia/nemotron-3-super-120b-a12b","usage":{"input":14960,"output":77,"cacheRead":0,"cacheWrite":0,"totalTokens":15037,"cost":{"input":0,"output":0,"cacheRead":0,"cacheWrite":0,"total":0}},"stopReason":"stop","timestamp":1775806780473,"responseId":"chatcmpl-9269a85cc94ddd79"}}
```

Here's a summary of the interaction.

1. User Request
Aaron Mitchell (phone: +1 (204) 452-6452) sent a message asking for invoice line IDs related to the artist Led Zeppelin.
2. Assistant thinks
The assistant reasoned internally that it needed to find the right tool, identified the music-store-assistant skill, and decided to read its SKILL.md file first.
3. Assistant reads the skill file
The assistant called the read tool on the SKILL.md file to understand how to use the music-store-assistant.
4. Skill instructions retrieved
The SKILL.md revealed that queries should be sent via a curl POST request to http://localhost:8001/generate with the user's message passed as input_message in the request body.
5. First API call attempt (failed)
The assistant made a curl request but used $'...' bash quoting syntax, which caused a JSON parsing error — the API responded with a JSON decode error.
6. Second API call attempt (succeeded)
The assistant corrected the quoting, using standard double quotes instead, and resubmitted the same request. This time the server processed it successfully.
7. API response received
The music store API returned the result: invoice line IDs 267 and 268 are associated with Led Zeppelin purchases for Aaron Mitchell.
8. Final response to user
The assistant relayed the result clearly, stating that the invoice line IDs related to Led Zeppelin for Aaron Mitchell are 267 and 268.

# NemoClaw

NemoClaw is an open-source stack from NVIDIA that aims to solve the security, privacy, and trust gap in running autonomous AI agents, specifically OpenClaw always-on assistants.  
The table below provides an overview of what NemoClaw aims to address.

| Aspect                  | OpenClaw baseline                                                                                                         | NemoClaw on top of OpenClaw                                                                                                    |
| ----------------------- | ------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| System access           | Full shell, file, and script access by default; often runs with broad privileges                         | Runs inside a sandboxed runtime with limited, policy‑controlled capabilities                              |
| Network exposure        | Many instances directly exposed to internet with weak auth and unsafe defaults                               | Designed for hardened deployments with controlled network egress and ingress via policy                         |
| Skill ecosystem         | Large number of unvetted community skills, with documented malware and harmful instructions                      | Does not “fix” skills, but reduces what any skill can do via sandbox and per‑skill policies                    |
| Prompt injection impact | Can lead to arbitrary code execution, credential leaks, and data exfiltration with few built‑in mitigations | Privacy router and guardrails can detect/block sensitive data flows triggered by malicious prompts          |
| Secrets & data handling | History of plaintext key leakage, exposed logs, and unclear data‑flow controls                            | Centralized policies over what data can leave the sandbox; monitoring of sensitive access attempts         |
| Enterprise controls     | Limited native support for audit, least privilege, and compliance controls                                    | Adds audit trails, configurable security/privacy policies, and integration into enterprise security tooling |

Find out more about how NemoClaw works [here](https://docs.nvidia.com/nemoclaw/latest/about/how-it-works.html).

<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 33%; text-align: left;"><a href="06_challenge.ipynb">Previous Notebook</a></span>
    <span style="float: left; width: 34%; text-align: center;">
        <a href="01_inference_endpoint.ipynb">1</a>
        <a href="02_introduction_mcp.ipynb">2</a>
        <a href="03_low_level_mcp.ipynb">3</a>
        <a href="04_langraph_agent.ipynb">4</a>
        <a href="05_nemo_agent_toolkit.ipynb">5</a>
        <a href="06_challenge.ipynb">6</a>
        <a >7</a>
    </span>
    <span style="float: left; width: 33%; text-align: right;"></span>
</div>